# Explore reviewed anesthesia EEG

This collaborator template uses GUI dialogs to select a participant HDF5 package and then select one recording inside it. It summarizes metadata and annotations, displays selectable EEG windows, and estimates clean-channel power spectra.

Important: each saved section is resampled to 250 Hz, 0.5–50 Hz filtered, common-average referenced, and then globally z-scored with one mean and standard deviation across all EEG channels and unmarked samples in that section. Signal values are dimensionless global z-scores; PSD units are **global z-score²/Hz**.

The loader supports both the current single-recording file and a future participant file containing multiple recordings in separate HDF5 groups.

## 1. Imports and file selection

If an import is missing, run `%pip install h5py numpy pandas scipy plotly PyQt6` in a separate cell, restart the kernel, and rerun the notebook.

In [ ]:
%pip install pandas

In [23]:
from pathlib import Path
import json

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy.signal import welch
from PyQt6.QtWidgets import QApplication, QFileDialog, QInputDialog

# Open Plotly figures in the normal web browser. This avoids the
# VS Code/Jupyter nbformat MIME-rendering error.
pio.renderers.default = "browser"

# Change this starting folder if the reviewed packages are moved.
DATA_FOLDER = Path(r"C:\Users\zhouz\Downloads\20260626\reorg_data")

# Keep a persistent Qt application for the GUI selection windows.
QT_APP = QApplication.instance() or QApplication([])

selected_package, _ = QFileDialog.getOpenFileName(
    None,
    "Select a participant reviewed EEG package",
    str(DATA_FOLDER),
    "Participant EEG package (*_reviewed_eeg.h5);;HDF5 files (*.h5)",
)

if not selected_package:
    raise RuntimeError("No participant package was selected.")

H5_FILE = Path(selected_package)

print("Opening:", H5_FILE)
print("File size (MB):", round(H5_FILE.stat().st_size / 1_000_000, 2))

Opening: C:\Users\zhouz\OneDrive - UC Irvine\Documents\Anes_Liveamp8_Data\Reorganized_liveamp8_data\L028_reviewed_eeg.h5
File size (MB): 16.65


## 2. Inspect the HDF5 structure

This shows which groups, datasets, dimensions, and attributes are stored without printing the EEG samples.

In [24]:
def print_hdf5_tree(file_path):
    with h5py.File(file_path, "r") as h5_file:
        print("Root attributes:")
        for key, value in h5_file.attrs.items():
            print(f"  {key}: {value}")

        print("\nGroups and datasets:")

        def visitor(name, item):
            indent = "  " * name.count("/")
            if isinstance(item, h5py.Dataset):
                print(
                    f"{indent}- {name} | dataset | "
                    f"shape={item.shape} | dtype={item.dtype}"
                )
            else:
                print(f"{indent}+ {name} | group")

        h5_file.visititems(visitor)

print_hdf5_tree(H5_FILE)

Root attributes:
  format_name: Anesthesia EEG participant package
  n_recordings: 5
  participant_id: L028

Groups and datasets:
+ recordings | group
  + recordings/or1 | group
    + recordings/or1/annotations | group
      - recordings/or1/annotations/channels | dataset | shape=(3,) | dtype=object
      - recordings/or1/annotations/duration_sec | dataset | shape=(3,) | dtype=float64
      - recordings/or1/annotations/label | dataset | shape=(3,) | dtype=object
      - recordings/or1/annotations/onset_sec | dataset | shape=(3,) | dtype=float64
    + recordings/or1/eeg | group
      - recordings/or1/eeg/channel_labels | dataset | shape=(8,) | dtype=object
      - recordings/or1/eeg/data_average_referenced_global_zscore | dataset | shape=(8, 59481) | dtype=float32
      - recordings/or1/eeg/globally_bad_channels | dataset | shape=(0,) | dtype=object
    + recordings/or1/review | group
      - recordings/or1/review/processing_description | dataset | shape=() | dtype=object
  + recordings

## 3. Load all recordings and embedded metadata

Current files contain one recording at the root. Future participant files may contain `/recordings/<recording_name>/...`. Both layouts are handled here.

In [26]:
def decode_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def decode_text_array(dataset):
    return [decode_text(value) for value in dataset[()]]


def attributes_to_dict(attributes):
    result = {}
    for key, value in attributes.items():
        if isinstance(value, np.generic):
            value = value.item()
        result[key] = decode_text(value) if isinstance(value, bytes) else value
    return result


def load_json_dataset(group, name):
    if group is None or name not in group:
        return []
    value = group[name][()]
    return json.loads(decode_text(value))


def load_recording(container, root_attributes, recording_name):
    eeg_group = container["eeg"]
    annotation_group = container.get("annotations")
    review_group = container.get("review")

    metadata = dict(root_attributes)
    metadata.update(attributes_to_dict(container.attrs))
    metadata.update(attributes_to_dict(eeg_group.attrs))

    channel_labels = decode_text_array(eeg_group["channel_labels"])
    globally_bad = (
        decode_text_array(eeg_group["globally_bad_channels"])
        if "globally_bad_channels" in eeg_group
        else []
    )

    if annotation_group is None:
        annotations = pd.DataFrame(
            columns=["onset_sec", "duration_sec", "label", "channels"]
        )
    else:
        annotations = pd.DataFrame(
            {
                "onset_sec": annotation_group["onset_sec"][()].astype(float),
                "duration_sec": annotation_group["duration_sec"][()].astype(float),
                "label": decode_text_array(annotation_group["label"]),
                "channels": decode_text_array(annotation_group["channels"]),
            }
        ).sort_values("onset_sec", ignore_index=True)

    return {
        "name": recording_name,
        "data": eeg_group["data_average_referenced_global_zscore"][()].astype(np.float32),
        "sampling_rate": float(eeg_group.attrs["sampling_frequency_hz"]),
        "channel_labels": channel_labels,
        "globally_bad_channels": globally_bad,
        "annotations": annotations,
        "metadata": metadata,
        "label_audit": load_json_dataset(review_group, "label_audit_json"),
        "channel_audit": load_json_dataset(review_group, "channel_audit_json"),
        "deleted_annotations": load_json_dataset(
            review_group,
            "deleted_annotations_json",
        ),
    }


with h5py.File(H5_FILE, "r") as h5_file:
    root_attributes = attributes_to_dict(h5_file.attrs)

    if "recordings" in h5_file:
        recording_names = list(h5_file["recordings"].keys())
        if not recording_names:
            raise ValueError("The selected participant package has no recordings.")

        selected_recording, accepted = QInputDialog.getItem(
            None,
            "Select a recording",
            "Recording to inspect:",
            recording_names,
            0,
            False,
        )

        if not accepted or not selected_recording:
            raise RuntimeError("No recording was selected.")

        RECORDING_NAME = str(selected_recording)
        recording = load_recording(
            h5_file["recordings"][RECORDING_NAME],
            root_attributes,
            RECORDING_NAME,
        )
    else:
        RECORDING_NAME = str(
            root_attributes.get("recording_id", H5_FILE.stem)
        )
        recording = load_recording(
            h5_file,
            root_attributes,
            RECORDING_NAME,
        )

# Only the selected recording is loaded into memory.
    recordings = {RECORDING_NAME: recording}

print("Selected package:", H5_FILE)
print("Loaded recording:", RECORDING_NAME)

Selected package: C:\Users\zhouz\OneDrive - UC Irvine\Documents\Anes_Liveamp8_Data\Reorganized_liveamp8_data\L028_reviewed_eeg.h5
Loaded recording: or1


## 4. Selected recording summary

In [27]:
summary_rows = []

for name, item in recordings.items():
    duration_sec = item["data"].shape[1] / item["sampling_rate"]
    summary_rows.append(
        {
            "recording": name,
            "context": item["metadata"].get("file_context", ""),
            "channels": item["data"].shape[0],
            "samples": item["data"].shape[1],
            "sampling_rate_hz": item["sampling_rate"],
            "duration_min": duration_sec / 60.0,
            "annotations": len(item["annotations"]),
            "globally_bad": ", ".join(item["globally_bad_channels"]),
        }
    )

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

print("Selected recording:", RECORDING_NAME)
print("Channel labels:", recording["channel_labels"])
print("Globally bad channels:", recording["globally_bad_channels"] or "none")

,recording,context,channels,samples,sampling_rate_hz,duration_min,annotations,globally_bad
0,or1,,8,59481,250.0,3.9654,3,


Selected recording: or1
Channel labels: ['Fp1', 'Fp2', 'F3', 'F4', 'P3', 'P4', 'O1', 'O2']
Globally bad channels: none


## 5. Review annotations

`channels = all` means the annotation applies to every channel. Otherwise, the value contains a comma-separated list of affected channels.

In [28]:
annotations = recording["annotations"]

print("Annotation counts by label:")
display(
    annotations["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

display(annotations.head(30))

Annotation counts by label:


,label,count
0,BAD_channel,1
1,fentanyl_bolus,1
2,propofol_bolus,1


,onset_sec,duration_sec,label,channels
0,0.954,233.916,BAD_channel,P3
1,87.820,0.001,fentanyl_bolus,all
2,185.000,0.001,propofol_bolus,all


## 6. Plot a selected EEG window

Change `START_SEC`, `DURATION_SEC`, or `CHANNELS_TO_PLOT`. This is an interactive Plotly figure. The displayed values are limited to ±3 z for readability; the stored EEG is not clipped.

In [ ]:
def plot_eeg_window(
    recording_item,
    start_sec=0.0,
    duration_sec=60.0,
    channels=None,
    display_global_z_limit=None,
    channel_spacing=None,
    maximum_points=15000,
):
    data = recording_item["data"]
    labels = recording_item["channel_labels"]
    sampling_rate = recording_item["sampling_rate"]

    if channels is None:
        channels = labels

    missing = [channel for channel in channels if channel not in labels]
    if missing:
        raise ValueError(f"Unknown channels: {missing}")

    channel_indices = [labels.index(channel) for channel in channels]
    start_sample = max(0, int(round(start_sec * sampling_rate)))
    end_sample = min(
        data.shape[1],
        int(round((start_sec + duration_sec) * sampling_rate)),
    )
    if end_sample <= start_sample:
        raise ValueError("The requested window is outside the recording.")

    display_step = max(
        1,
        int(np.ceil((end_sample - start_sample) / maximum_points)),
    )
    samples = np.arange(start_sample, end_sample, display_step)
    time_sec = samples / sampling_rate
    window_values = data[np.ix_(channel_indices, samples)]
    if display_global_z_limit is None:
        display_global_z_limit = float(
            np.nanpercentile(np.abs(window_values), 98.0)
        )
    if not np.isfinite(display_global_z_limit) or display_global_z_limit == 0:
        display_global_z_limit = 1.0
    if channel_spacing is None:
        channel_spacing = 2.5 * display_global_z_limit
    offsets = np.arange(len(channels))[::-1] * channel_spacing

    figure = go.Figure()
    colors = [
        "#0072B2", "#D55E00", "#009E73", "#CC79A7",
        "#E69F00", "#56B4E9", "#222222", "#7B2CBF",
    ]

    for row, (channel, channel_index) in enumerate(
        zip(channels, channel_indices)
    ):
        values = np.clip(
            data[channel_index, samples],
            -display_global_z_limit,
            display_global_z_limit,
        )
        figure.add_trace(
            go.Scattergl(
                x=time_sec,
                y=values + offsets[row],
                mode="lines",
                name=channel,
                line={"width": 1.2, "color": colors[row % len(colors)]},
                customdata=values,
                hovertemplate=(
                    f"<b>{channel}</b><br>"
                    "Time: %{x:.3f} sec<br>"
                    "Global z-score: %{customdata:.2f}<extra></extra>"
                ),
            )
        )

    window_annotations = recording_item["annotations"].loc[
        lambda frame: (
            (frame["onset_sec"] <= end_sample / sampling_rate)
            & (
                frame["onset_sec"] + frame["duration_sec"]
                >= start_sample / sampling_rate
            )
        )
    ]

    top = offsets[0] + display_global_z_limit
    bottom = offsets[-1] - display_global_z_limit

    for _, annotation in window_annotations.iterrows():
        onset = float(annotation["onset_sec"])
        duration = float(annotation["duration_sec"])
        label = str(annotation["label"])
        affected = str(annotation["channels"])

        if duration > 0:
            fill = (
                "rgba(255,0,0,0.14)"
                if label.startswith("BAD_")
                else "rgba(30,100,255,0.09)"
            )
            figure.add_vrect(
                x0=onset,
                x1=onset + duration,
                fillcolor=fill,
                line_width=0,
                layer="below",
            )

        figure.add_vline(
            x=onset,
            line_color="red",
            line_dash="dash",
            line_width=1,
        )
        figure.add_annotation(
            x=onset,
            y=top + 0.5,
            text=f"{label} [{affected}]",
            showarrow=False,
            textangle=-45,
            yanchor="top",
            font={"color": "red", "size": 10},
        )

    figure.update_layout(
        title=f"{recording_item['name']}: reviewed EEG",
        xaxis_title="Time from recording start (seconds)",
        yaxis={
            "tickmode": "array",
            "tickvals": offsets,
            "ticktext": channels,
            "range": [bottom - 1, top + 2],
        },
        height=max(650, 100 * len(channels)),
        showlegend=False,
        hovermode="closest",
        plot_bgcolor="white",
        margin={"l": 100, "r": 30, "t": 80, "b": 60},
    )
    figure.update_xaxes(showgrid=True, gridcolor="rgba(180,180,180,0.25)")
    figure.update_yaxes(showgrid=False)
    figure.show(renderer="browser")


# Interactive browser viewer: choose an interval on the full recording,
# focus it, then inspect it with a shared display gain.
import importlib.util
import socket
import threading
import webbrowser

if importlib.util.find_spec("dash") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dash"])

from dash import Dash, Input, Output, State, ctx, dcc, html, no_update
from werkzeug.serving import make_server

EXPLORE_MAX_POINTS = 20000
EXPLORE_GAIN = 1.0
EXPLORE_FOCUS = None
EXPLORE_DISPLAY_STEP = max(1, int(np.ceil(recording["data"].shape[1] / EXPLORE_MAX_POINTS)))
EXPLORE_SAMPLES = np.arange(0, recording["data"].shape[1], EXPLORE_DISPLAY_STEP)
EXPLORE_TIME_MINUTES = EXPLORE_SAMPLES / recording["sampling_rate"] / 60.0
EXPLORE_VALUES = recording["data"][:, EXPLORE_SAMPLES].astype(np.float32)
EXPLORE_LIMIT = float(np.nanpercentile(np.abs(EXPLORE_VALUES), 98.0)) or 1.0
EXPLORE_VALUES = np.clip(EXPLORE_VALUES, -EXPLORE_LIMIT, EXPLORE_LIMIT)
EXPLORE_LABELS = recording["channel_labels"]
EXPLORE_OFFSETS = np.arange(len(EXPLORE_LABELS))[::-1] * (2.5 * EXPLORE_LIMIT)


def build_explorer_figure():
    gain = EXPLORE_GAIN
    fig = go.Figure()
    colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#222222", "#7B2CBF"]
    for index, label in enumerate(EXPLORE_LABELS):
        fig.add_trace(go.Scattergl(
            x=EXPLORE_TIME_MINUTES,
            y=EXPLORE_VALUES[index] * gain + EXPLORE_OFFSETS[index],
            mode="lines", name=label, line={"width": 1.1, "color": colors[index % len(colors)]},
            customdata=EXPLORE_VALUES[index],
            hovertemplate=f"<b>{label}</b><br>Time: %{{x:.4f}} min<br>Global z-score: %{{customdata:.2f}}<extra></extra>",
        ))
    top = EXPLORE_OFFSETS[0] + 1.6 * EXPLORE_LIMIT
    for _, row in recording["annotations"].iterrows():
        onset = float(row["onset_sec"]) / 60.0
        duration = float(row["duration_sec"]) / 60.0
        if EXPLORE_FOCUS and (onset + duration < EXPLORE_FOCUS[0] or onset > EXPLORE_FOCUS[1]):
            continue
        label = str(row["label"])
        affected_text = str(row["channels"])
        affected_channels = [] if affected_text.casefold() == "all" else [value.strip() for value in affected_text.split(",") if value.strip()]
        if duration > 0:
            fill = "rgba(255,0,0,0.14)" if label.startswith("BAD_") else "rgba(30,100,255,0.09)"
            if label == "BAD_channel" and affected_channels:
                # Match the organizing display: shade only the named channel rows.
                for affected_channel in affected_channels:
                    if affected_channel not in EXPLORE_LABELS:
                        continue
                    channel_index = EXPLORE_LABELS.index(affected_channel)
                    spacing = 2.5 * EXPLORE_LIMIT
                    fig.add_shape(type="rect", xref="x", yref="y", x0=onset, x1=onset + duration, y0=EXPLORE_OFFSETS[channel_index] - .42 * spacing, y1=EXPLORE_OFFSETS[channel_index] + .42 * spacing, fillcolor=fill, line_width=0, layer="below")
            else:
                fig.add_vrect(x0=onset, x1=onset + duration, fillcolor=fill, line_width=0, layer="below")
        fig.add_vline(x=onset, line_color="crimson", line_dash="dash", line_width=1)
        label_text = f"{label} [{','.join(affected_channels)}]" if label == "BAD_channel" and affected_channels else label
        fig.add_annotation(x=onset, y=top, text=label_text, showarrow=False, textangle=-45, font={"color": "crimson", "size": 10})
    fig.update_layout(
        title=f"{recording['name']}: reviewed global z-score EEG",
        xaxis={"title": "Minutes from recording start", "range": list(EXPLORE_FOCUS) if EXPLORE_FOCUS else None, "rangeslider": {"visible": True, "thickness": 0.06}},
        yaxis={"tickmode": "array", "tickvals": EXPLORE_OFFSETS, "ticktext": EXPLORE_LABELS, "range": [EXPLORE_OFFSETS[-1] - 1.2 * EXPLORE_LIMIT, EXPLORE_OFFSETS[0] + 2.1 * EXPLORE_LIMIT]},
        height=max(420, 40 * len(EXPLORE_LABELS)), showlegend=False, dragmode="select", hovermode="closest", plot_bgcolor="white", margin={"l": 100, "r": 25, "t": 65, "b": 55}, uirevision="explore-view",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(180,180,180,.25)", minor={"dtick": 1/60, "showgrid": True, "gridcolor": "rgba(110,110,110,.35)", "griddash": "dash"})
    return fig


explore_app = Dash(__name__)
explore_app.layout = html.Div([
    html.Div([
        html.Button("Focus selected interval", id="explore-focus", n_clicks=0),
        html.Button("Show full recording", id="explore-full", n_clicks=0, style={"marginLeft": "6px"}),
        html.Button("− gain", id="explore-gain-down", n_clicks=0, style={"marginLeft": "14px"}),
        html.Button("+ gain", id="explore-gain-up", n_clicks=0, style={"marginLeft": "6px"}),
        html.Button("Reset gain", id="explore-gain-reset", n_clicks=0, style={"marginLeft": "6px"}),
        html.Span(id="explore-status", children="Drag across the EEG to select an interval.", style={"marginLeft": "14px"}),
    ], style={"padding": "8px"}),
    dcc.Store(id="explore-selection"),
    dcc.Graph(id="explore-graph", figure=build_explorer_figure(), config={"displaylogo": False, "scrollZoom": True, "responsive": True}),
], style={"fontFamily": "Arial", "padding": "10px"})


@explore_app.callback(Output("explore-selection", "data"), Output("explore-status", "children"), Input("explore-graph", "selectedData"), prevent_initial_call=True)
def capture_explorer_selection(selected):
    if not selected:
        return no_update, "Drag across the EEG to select an interval."
    values = selected.get("range", {}).get("x")
    if not values:
        return no_update, "No interval was selected."
    start, end = sorted(float(value) for value in values)
    return [start, end], f"Selected {start:.4f}–{end:.4f} minutes. Click Focus selected interval."


@explore_app.callback(Output("explore-graph", "figure"), Output("explore-status", "children", allow_duplicate=True), Input("explore-focus", "n_clicks"), Input("explore-full", "n_clicks"), Input("explore-gain-down", "n_clicks"), Input("explore-gain-up", "n_clicks"), Input("explore-gain-reset", "n_clicks"), State("explore-selection", "data"), prevent_initial_call=True)
def update_explorer_view(focus, full, down, up, reset, selection):
    global EXPLORE_FOCUS, EXPLORE_GAIN
    triggered = ctx.triggered_id
    if triggered == "explore-focus":
        if not selection or selection[1] <= selection[0]:
            return no_update, "Select a non-zero interval first."
        EXPLORE_FOCUS = tuple(selection)
        message = f"Focused {selection[0]:.4f}–{selection[1]:.4f} minutes."
    elif triggered == "explore-full":
        EXPLORE_FOCUS = None
        message = "Showing full recording."
    elif triggered == "explore-gain-up":
        EXPLORE_GAIN *= 2.0
        message = f"Display gain: {EXPLORE_GAIN:.2f}×"
    elif triggered == "explore-gain-down":
        EXPLORE_GAIN = max(.01, EXPLORE_GAIN / 2.0)
        message = f"Display gain: {EXPLORE_GAIN:.2f}×"
    else:
        EXPLORE_GAIN = 1.0
        message = "Display gain reset."
    return build_explorer_figure(), message


with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    EXPLORE_PORT = sock.getsockname()[1]
EXPLORE_URL = f"http://127.0.0.1:{EXPLORE_PORT}"
EXPLORE_SERVER = make_server("127.0.0.1", EXPLORE_PORT, explore_app.server, threaded=True)
threading.Thread(target=EXPLORE_SERVER.serve_forever, daemon=True).start()
webbrowser.open(EXPLORE_URL)
print("Interactive interval viewer:", EXPLORE_URL)


Interactive interval viewer: http://127.0.0.1:59983


127.0.0.1 - - [01/Sep/2026 20:39:48] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_1m1785032845.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/deps/react@18.v4_4_1m1785032845.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_4_1m1785032845.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_1m1785032845.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_1m1785032845.min.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_4_1m1785032845.js HTTP/1.1" 200 -
127.0.0.1 - - [01/Sep/2026 20:39:48] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_4_1m1785032845.js HTTP/

## 7. Calculate clean-data availability

A channel is excluded completely if it is globally bad. Channel-specific `BAD_` annotations remove samples only from the listed channel; `channels = all` removes the interval from every channel.

In [ ]:
def bad_sample_mask(recording_item, channel):
    number_samples = recording_item["data"].shape[1]
    sampling_rate = recording_item["sampling_rate"]
    mask = np.zeros(number_samples, dtype=bool)

    if channel in recording_item["globally_bad_channels"]:
        mask[:] = True
        return mask

    for _, annotation in recording_item["annotations"].iterrows():
        label = str(annotation["label"])
        if not label.startswith("BAD_"):
            continue

        channel_text = str(annotation["channels"])
        affected_channels = {
            value.strip()
            for value in channel_text.split(",")
            if value.strip()
        }
        applies_to_channel = (
            channel_text.casefold() == "all"
            or channel in affected_channels
        )
        if not applies_to_channel:
            continue

        start = max(
            0,
            int(np.floor(float(annotation["onset_sec"]) * sampling_rate)),
        )
        end = min(
            number_samples,
            int(
                np.ceil(
                    (
                        float(annotation["onset_sec"])
                        + float(annotation["duration_sec"])
                    )
                    * sampling_rate
                )
            ),
        )
        if end <= start:
            end = min(number_samples, start + 1)
        mask[start:end] = True

    return mask


clean_summary = []
for channel in recording["channel_labels"]:
    mask = bad_sample_mask(recording, channel)
    clean_summary.append(
        {
            "channel": channel,
            "globally_bad": channel in recording["globally_bad_channels"],
            "clean_minutes": (~mask).sum() / recording["sampling_rate"] / 60,
            "clean_percent": 100 * (~mask).mean(),
        }
    )

clean_summary_table = pd.DataFrame(clean_summary)
display(clean_summary_table.round({"clean_minutes": 2, "clean_percent": 1}))

## 8. Estimate PSD from completely clean epochs

This divides each channel into fixed-length epochs and uses only epochs that do not overlap a channel-relevant `BAD_` annotation. Globally bad channels are skipped. PSD units are z²/Hz.

In [ ]:
def clean_epoch_psd(recording_item, channel, epoch_sec=10.0):
    if channel in recording_item["globally_bad_channels"]:
        return None, None, 0

    labels = recording_item["channel_labels"]
    channel_index = labels.index(channel)
    data = recording_item["data"][channel_index]
    sampling_rate = recording_item["sampling_rate"]
    bad_mask = bad_sample_mask(recording_item, channel)
    epoch_samples = max(2, int(round(epoch_sec * sampling_rate)))

    epoch_psds = []
    frequencies = None

    for start in range(0, len(data) - epoch_samples + 1, epoch_samples):
        end = start + epoch_samples
        if bad_mask[start:end].any():
            continue

        frequencies, power = welch(
            data[start:end],
            fs=sampling_rate,
            nperseg=min(epoch_samples, int(round(4 * sampling_rate))),
            noverlap=None,
            detrend="constant",
            scaling="density",
        )
        epoch_psds.append(power)

    if not epoch_psds:
        return None, None, 0

    return frequencies, np.mean(epoch_psds, axis=0), len(epoch_psds)


psd_figure = go.Figure()
psd_summary = []

for channel in recording["channel_labels"]:
    frequencies, power, number_epochs = clean_epoch_psd(
        recording,
        channel,
        epoch_sec=10.0,
    )
    psd_summary.append(
        {"channel": channel, "clean_10s_epochs": number_epochs}
    )
    if number_epochs == 0:
        continue

    frequency_mask = (frequencies >= 0.5) & (frequencies <= 50.0)
    psd_figure.add_trace(
        go.Scatter(
            x=frequencies[frequency_mask],
            y=10 * np.log10(power[frequency_mask] + np.finfo(float).tiny),
            mode="lines",
            name=channel,
        )
    )

psd_figure.update_layout(
    title=f"{RECORDING_NAME}: mean PSD from clean 10-second epochs",
    xaxis_title="Frequency (Hz)",
    yaxis_title="PSD (dB z²/Hz)",
    template="plotly_white",
)
psd_figure.show(renderer="browser")

display(pd.DataFrame(psd_summary))

## 9. Review embedded audit information

In [ ]:
print("File and processing metadata:")
display(pd.Series(recording["metadata"], name="value").to_frame())

print("Channel audit:")
display(pd.DataFrame(recording["channel_audit"]))

print("Label audit (first 30 rows):")
display(pd.DataFrame(recording["label_audit"]).head(30))

print("Annotations deleted during review (first 30 rows):")
display(pd.DataFrame(recording["deleted_annotations"]).head(30))

## Recommended future multi-recording layout

Multiple recordings can be stored in one participant HDF5 file, but they should remain separate groups rather than being concatenated. Durations, sampling rates, channel availability, and gaps may differ.

```text
L000_reviewed_eeg.h5
├── participant metadata
└── recordings
    ├── preop1_run-01
    │   ├── eeg
    │   ├── annotations
    │   └── review
    ├── or1_run-02
    │   ├── eeg
    │   ├── annotations
    │   └── review
    └── or2_run-03
        ├── eeg
        ├── annotations
        └── review
```

This notebook will automatically recognize that layout. Keep the original BrainVision recordings and a backup of each reviewed export; one corrupted participant file should not become the only copy of all recordings.